In [4]:
from sentence_transformers import SentenceTransformer, InputExample, losses
from torch.utils.data import DataLoader
import random
import pandas as pd

random.seed(42)

df_clean = pd.read_csv("../data/processed/cleaned_dataset.csv")

# Step 1: Prepare training pairs
high_texts = df_clean[df_clean["target"] == 1]["text"].tolist()
low_texts = df_clean[df_clean["target"] == 0]["text"].tolist()

# Sample 2000 from each
high_sample = random.sample(high_texts, 2000)
low_sample = random.sample(low_texts, 2000)

train_examples = []

# Positive pairs — both High
for text in high_sample:
    pair = random.choice(high_sample)
    train_examples.append(InputExample(texts=[text, pair], label=1.0))

# Negative pairs — High vs Low
for text in high_sample:
    pair = random.choice(low_sample)
    train_examples.append(InputExample(texts=[text, pair], label=0.0))

# Positive pairs — both Low
for text in low_sample:
    pair = random.choice(low_sample)
    train_examples.append(InputExample(texts=[text, pair], label=1.0))

print("Total training pairs:", len(train_examples))

# Step 2: Fine-tune
model_ft = SentenceTransformer("all-MiniLM-L6-v2")

train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=32)

train_loss = losses.CosineSimilarityLoss(model_ft)

model_ft.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=3,
    warmup_steps=100,
    output_path="models/finetuned_embedding_model",
    show_progress_bar=True,
)

print("Fine-tuning complete.")


Total training pairs: 6000


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6067.09it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Step,Training Loss
500,0.235862


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  4.93it/s]


Fine-tuning complete.
